# Уравнение Лапласа: 

## $\nabla^2\varphi = 0$

# Уравнение Пуассона:

## $\nabla^2\varphi = -\frac{\rho}{\varepsilon_0}$

## $\varphi$ - Электростатический потенциал (Вольт)

## $\rho$ - Объемная плотность заряда ($\frac{Кл}{м^3}$)

## $\varepsilon_0 = 8.86*10^{-12} \frac{Ф}{м}$ - Диэлектрическая проницаемость вакуума

## $E = -\nabla{\varphi}$ - Вектор напряженности электрического поля

# Итерационные методы Гаусса-Зейделя и Якоби

Система уравнений:

## $Ax = b$

Представим матрицу $A$ в виде $A = D + L + U$,

$D $ - диагональная матрица

$L$  - нижнетреугольная матрица (без диагонали)

$U $ - верхнетреугольная матрица (без диагонали)

Тогда для метода Якоби итерационный процесс выглядит как:

$x_{k+1}=D^{-1}(b−(L+U)x_{k}$

В методе Гаусса-Зейделя:

$x_{k+1}=(D+L)^{-1}(b−Ux_{k})$

Условие сходимости:

1. $||x_{k+1}-x_{k}||<=\varepsilon$
2. $||Ax_{k}-b||<=\varepsilon$


## Граничные условия (незаряженный проводящий квадрат)

### 1. $\varphi = 0$ - на всей поверхности квадрата (заземленный)

### 2. $\varphi = \varphi_c = const$ (незаряженный)

### $\varphi_c$ - неизвестно и определяется из условия отсутствия суммарного заряда

### $\oint_{S}\varepsilon_0\frac{\partial{\varphi}}{\partial{n}}dS = 0$

In [ ]:
# Импорт необходимых библиотек
import numpy as np
import matplotlib.pyplot as plt
import time

In [ ]:
# Задание геометрии задачи
def SetupGeometry(nx=121, ny=61, V=1.0, plate_thickness=1):
    phi = np.zeros((ny, nx), dtype=float)
    dirichlet_mask = np.zeros_like(phi, dtype=bool)
    dirichlet_values = np.zeros_like(phi, dtype=float)
    y0 = 2
    dirichlet_mask[y0:y0+plate_thickness, :] = True
    dirichlet_values[y0:y0+plate_thickness, :] = V
    dirichlet_mask[-(plate_thickness+2):-(2), :] = True
    dirichlet_values[-(plate_thickness+2):-(2), :] = -V
    dirichlet_mask[:, 0] = True; dirichlet_values[:, 0] = 0.0
    dirichlet_mask[:, -1] = True; dirichlet_values[:, -1] = 0.0
    dirichlet_mask[0, :] = True; dirichlet_values[0, :] = 0.0
    dirichlet_mask[-1, :] = True; dirichlet_values[-1, :] = 0.0
    phi[dirichlet_mask] = dirichlet_values[dirichlet_mask]
    return phi, dirichlet_mask, dirichlet_values

In [ ]:
# Вычисляет невязку (L2-норма лапласиана в свободных точках)
def Residuals(phi, fixed_mask):
    lap = (-4*phi + np.roll(phi,1,0) + np.roll(phi,-1,0) + np.roll(phi,1,1) + np.roll(phi,-1,1))
    r = lap[~fixed_mask]
    return float(np.sqrt(np.mean(r*r)))

In [ ]:
# Возвращает средний потенциал на внешней границе проводника
def AveragePhi(phi, mask):
    ny, nx = phi.shape
    halo_vals = []
    ys, xs = np.where(mask)
    for y, x in zip(ys, xs):
        if y>0 and not mask[y-1, x]:
            halo_vals.append(phi[y-1, x])
        if y<ny-1 and not mask[y+1, x]:
            halo_vals.append(phi[y+1, x])
        if x>0 and not mask[y, x-1]:
            halo_vals.append(phi[y, x-1])
        if x<nx-1 and not mask[y, x+1]:
            halo_vals.append(phi[y, x+1])
    if len(halo_vals)==0:
        return 0.0
    return float(np.mean(halo_vals))

In [ ]:
#Метод Якоби
def Jacobi(phi0, dirichlet_mask, dirichlet_values, tol=1e-4, max_iter=10000, floating_mask=None):
    phi = phi0.copy()
    ny, nx = phi.shape
    free = ~dirichlet_mask
    res_hist = []
    t0 = time.time()
    for k in range(max_iter):
        phi_new = phi.copy()
        phi_new[1:-1,1:-1][free[1:-1,1:-1]] = 0.25 * (
            phi[:-2,1:-1] + phi[2:,1:-1] + phi[1:-1,:-2] + phi[1:-1,2:]
        )[free[1:-1,1:-1]]
        phi_new[dirichlet_mask] = dirichlet_values[dirichlet_mask]
        if floating_mask is not None:
            halo_val = AveragePhi(phi_new, floating_mask)
            phi_new[floating_mask] = halo_val
        phi = phi_new
        fixed_mask = dirichlet_mask | (floating_mask if floating_mask is not None else False)
        r = Residuals(phi, fixed_mask)
        res_hist.append(r)
        if r < tol:
            break
    t1 = time.time()
    return phi, k+1, res_hist, (t1 - t0)

In [ ]:
# Метод Гаусса-Зейделя
def GaussZeidel(phi0, dirichlet_mask, dirichlet_values, tol=1e-4, max_iter=10000, floating_mask=None):
    phi = phi0.copy()
    ny, nx = phi.shape
    res_hist = []
    free = ~dirichlet_mask
    t0 = time.time()
    for k in range(max_iter):
        for j in range(1, ny-1):
            for i in range(1, nx-1):
                if free[j, i] and not (floating_mask[j, i] if floating_mask is not None else False):
                    phi[j, i] = 0.25 * (phi[j-1, i] + phi[j+1, i] + phi[j, i-1] + phi[j, i+1])
        phi[dirichlet_mask] = dirichlet_values[dirichlet_mask]
        if floating_mask is not None:
            halo_val = AveragePhi(phi, floating_mask)
            phi[floating_mask] = halo_val
        fixed_mask = dirichlet_mask | (floating_mask if floating_mask is not None else False)
        r = Residuals(phi, fixed_mask)
        res_hist.append(r)
        if r < tol:
            break
    t1 = time.time()
    return phi, k+1, res_hist, (t1 - t0)

In [ ]:
# Вычисление электрического поля из потенциала
def E(phi):
    Ex = np.zeros_like(phi)
    Ey = np.zeros_like(phi)
    Ex[:,1:-1] = -(phi[:,2:] - phi[:,:-2]) / 2.0
    Ey[1:-1,:] = -(phi[2:,:] - phi[:-2,:]) / 2.0
    Ex[:,0] = -(phi[:,1] - phi[:,0])
    Ex[:,-1] = -(phi[:,-1] - phi[:,-2])
    Ey[0,:] = -(phi[1,:] - phi[0,:])
    Ey[-1,:] = -(phi[-1,:] - phi[-2,:])
    return Ex, Ey


In [ ]:
# Визуализация потенциала и электрического поля
def Vusialize(phi, title):
    Ex, Ey = E(phi)
    E_mag = np.sqrt(Ex**2 + Ey**2)
    Exn = Ex / (E_mag + 1e-12)
    Eyn = Ey / (E_mag + 1e-12)
    ny, nx = phi.shape
    X, Y = np.meshgrid(np.arange(nx), np.arange(ny))
    plt.figure(figsize=(7, 4.2))
    plt.imshow(phi, origin='lower', aspect='auto', cmap='coolwarm')
    plt.title(title)
    plt.colorbar(label='Потенциал φ ')
    plt.tight_layout()
    plt.show()
    step = max(1, min(nx, ny)//25)
    plt.figure(figsize=(7, 4.2))
    plt.imshow(E_mag, origin='lower', aspect='auto', cmap='inferno')
    plt.quiver(
        X[::step, ::step], Y[::step, ::step],
        Exn[::step, ::step], Eyn[::step, ::step],
        color='white', pivot='mid', scale=30, width=0.004
    )
    plt.title(title)
    plt.colorbar(label='Амплитуда |E|')
    plt.tight_layout()
    plt.show()

In [ ]:
# Сравнение сходимости методов
def Compare(res_j, res_gs, title="Сравнение методов"):
    plt.figure(figsize=(7,4.2))
    plt.semilogy(res_j, label='Jacobi')
    plt.semilogy(res_gs, label='Gauss-Seidel')
    plt.xlabel('Итерация')
    plt.ylabel('Невязка')
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Плоский конденсатор
phi0, dmask, dvals = SetupGeometry()
phi_start = phi0.copy()
phi_j, it_j, res_j, t_j = Jacobi(phi_start, dmask, dvals, tol=1e-4)
phi_gs, it_gs, res_gs, t_gs = GaussZeidel(phi_start, dmask, dvals, tol=1e-4)
print(f"Плоский конденсатор: [Jacobi] iters={it_j}, time={t_j:.3f}s, residual={res_j[-1]:.2e}")
print(f"Плоский конденсатор: [Gauss-Seidel] iters={it_gs}, time={t_gs:.3f}s, residual={res_gs[-1]:.2e}")
Vusialize(phi_gs, "Плоский конденсатор [Gauss-Seidel]")
Vusialize(phi_j, "Плоский конденсатор [Jacobi]")
Compare(res_j, res_gs, "Сравнение методов (Плоский конденсатор)")


In [ ]:

# Заземленный квадрат
phi0, dmask, dvals = SetupGeometry()
ny, nx = phi0.shape
size = min(nx, ny)//10
cy, cx = ny//2, nx//2
sq = np.zeros_like(phi0, dtype=bool)
sq[cy-size:cy+size+1, cx-size:cx+size+1] = True
dmask2 = dmask.copy(); dvals2 = dvals.copy()
dmask2[sq] = True; dvals2[sq] = 0.0
phi0[dmask2] = dvals2[dmask2]
phi_gs2, it_gs2, res_gs2, t_gs2 = GaussZeidel(phi0, dmask2, dvals2, tol=1e-4)
print(f"Заземленный квадрат: [Gauss-Seidel] iters={it_gs2}, time={t_gs2:.3f}s, residual={res_gs2[-1]:.2e}")
Vusialize(phi_gs2, "Заземленный квадрат [Gauss-Seidel]")
phi_start2 = phi0.copy()
phi_j2, it_j2, res_j2, t_j2 = Jacobi(phi_start2, dmask2, dvals2, tol=1e-4)
print(f"Заземленный квадрат: [Jacobi] iters={it_j2}, time={t_j2:.3f}s, residual={res_j2[-1]:.2e}")
Vusialize(phi_j2, "Заземленный квадрат [Jacobi]")
Compare(res_j2, res_gs2, "Сравнение методов (Заземленный квадрат)")




In [ ]:
# Незаряженный квадрат
phi0, dmask, dvals = SetupGeometry()
sq = np.zeros_like(phi0, dtype=bool)
sq[cy-size:cy+size+1, cx-size:cx+size+1] = True
phi0[sq] = 0.0
phi_gs3, it_gs3, res_gs3, t_gs3 = GaussZeidel(phi0, dmask, dvals, tol=1e-4, floating_mask=sq)
print(f"Незаряженный квадрат: [Gauss-Seidel] iters={it_gs3}, time={t_gs3:.3f}s, residual={res_gs3[-1]:.2e}")
Vusialize(phi_gs3, "Незаряженный квадрат [Gauss-Seidel]")
phi_start = phi0.copy()
phi_j3, it_j3, res_j3, t_j3 = Jacobi(phi_start, dmask, dvals, tol=1e-4, floating_mask=sq)
print(f"Незаряженный квадрат: [Jacobi] iters={it_j3}, time={t_j3:.3f}s, residual={res_j3[-1]:.2e}")
Vusialize(phi_j3, "Незаряженный квадрат [Jacobi]")   
Compare(res_j3, res_gs3, "Сравнение методов (Незаряженный квадрат)")